# minimal-react-harness — Colab 실행 노트북

온디바이스 **지원사업 추천 ReAct 에이전트**를 작은 모델(Qwen2.5-1.5B)로 실제로 돌려보고,
작은 모델이 도구 호출 형식을 어디서 깨뜨리는지 **관찰**합니다.

> **먼저 GPU 켜기**: 메뉴 `런타임 > 런타임 유형 변경 > 하드웨어 가속기 = T4 GPU` 선택.

실행 순서: 1) GPU 확인 → 2) clone → 3) 설치 → 4) 엔진 테스트(모델 불필요) → 5) 데이터 도구 확인 → 6) 실제 모델 추천 관찰.

## 1) GPU 확인
`cuda=True` 가 나와야 빠릅니다. False면 위 안내대로 런타임 유형을 T4로 바꾸세요.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
!nvidia-smi -L

## 2) 레포 clone
public 레포라 토큰 없이 그대로 clone 됩니다.

In [ ]:
%cd /content
![ -d minimal-react-harness ] && rm -rf minimal-react-harness
!git clone https://github.com/hyeminhassomething/minimal-react-harness.git
%cd minimal-react-harness
!ls

## 3) 의존성 설치
Colab에는 torch가 이미 있으니 transformers/accelerate/pandas만 설치합니다.

In [ ]:
!pip install -q "transformers>=4.40" "accelerate>=0.30" "pandas>=2.0"

## 4) 엔진 테스트 (모델 불필요)
MockLLM으로 하네스 로직 6개 테스트. 전부 PASS 여야 합니다.

In [ ]:
!python test_harness.py

## 5) 데이터 도구 확인 (자격 대조가 '코드'로 도는지)
모델 없이 `check_eligibility`를 직접 호출 — 소득 상한/나이/가구 판정이 코드로 이뤄짐.

In [ ]:
from tools import dataset_search, check_eligibility, _parse_profile
print(_parse_profile('서울 사는 27살 1인가구 월소득 250만원, 주거 지원'))
print('---')
print(check_eligibility('서울 27살 1인가구 월소득 250만원'))
print('--- 소득 초과 케이스 ---')
print(check_eligibility('전국 20살 월소득 300만원'))

## 6) 실제 작은 모델로 추천 관찰
Qwen2.5-1.5B 다운로드(~3GB) 후 추천 시나리오를 돌립니다. `--show-raw`로 모델 원문을 함께 봐서,
작은 모델이 `dataset_search`/`check_eligibility` 형식을 어디서 깨뜨리는지 관찰하세요.

In [ ]:
!python recommend.py -p "서울 사는 27살 1인가구 월소득 250만원, 주거 지원 알아봐줘" --show-raw --save

[env] torch=2.11.0+cu128 device=cuda
[model] Qwen/Qwen2.5-1.5B-Instruct 로드 중...
config.json: 100% 660/660 [00:00<00:00, 3.61MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 24.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 73.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 117MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 141MB/s]
model.safetensors: 100% 3.09G/3.09G [00:27<00:00, 111MB/s]
Loading weights: 100% 338/338 [00:03<00:00, 102.47it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.62MB/s]
입력: 서울 사는 27살 1인가구 월소득 250만원, 주거 지원 알아봐줘


### (선택) 에펠탑 토이 예제도 실행
기존 예제가 그대로 도는지 확인.

In [ ]:
!python main.py -q "에펠탑은 올해(2026년) 기준 몇 년 됐어?" --show-raw

## 관찰 → 수정 루프
형식 오류가 보이면, 로컬에서 `react_agent.py`(프롬프트/파서)를 한 군데 고치고 push → 이 노트북에서
`git pull` 후 6번 셀을 다시 실행해 `[형식 오류 N회]`가 어떻게 바뀌는지 비교하세요.

In [ ]:
!git pull  # 로컬에서 수정+push 한 뒤 실행